In [1]:
import yfinance as yf
import requests 
from io import StringIO
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import vectorbt as vbt
import talib
import warnings

warnings.filterwarnings("ignore")



In c:\Users\Haresh L\anaconda3\Lib\site-packages\matplotlib\mpl-data\stylelib\bmh.mplstyle: The text.hinting_factor rcParam was deprecated in Matplotlib 3.11 and will be removed in 3.13.
In c:\Users\Haresh L\anaconda3\Lib\site-packages\matplotlib\mpl-data\stylelib\classic.mplstyle: The text.hinting_factor rcParam was deprecated in Matplotlib 3.11 and will be removed in 3.13.
In c:\Users\Haresh L\anaconda3\Lib\site-packages\matplotlib\mpl-data\stylelib\_classic_test_patch.mplstyle: The text.kerning_factor rcParam was deprecated in Matplotlib 3.11 and will be removed in 3.13.


In [2]:
url = "https://en.wikipedia.org/wiki/List_of_S%26P_500_companies"
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/120.0.0.0 Safari/537.36"
    )
}

response = requests.get(url, headers=headers)
response.raise_for_status()  


table = pd.read_html(StringIO(response.text))
df_tickers = table[0]


tickers = df_tickers['Symbol'].str.replace('.', '-', regex=False).tolist()


tickers = tickers[:500]

df3 = yf.download(tickers, start= "2015-05-05", end= "2026-05-05", threads = True)

try:
  close_df = df3["Close"]
  high_df = df3["High"]
  low_df = df3["Low"]
  open_df = df3["Open"]
  volume_df = df3["volume"]

except KeyError:
  
  close_df = df3.xs("Close", level=0, axis=1)
  high_df = df3.xs("High", level=0, axis=1)
  low_df = df3.xs("Low", level=0, axis=1)
  open_df = df3.xs("Open", level = 0, axis = 1)
  volume_df = df3.xs("Volume", level = 0, axis = 1)

close_df = close_df.dropna(axis=1, how="all")
high_df = high_df.dropna(axis=1, how="all")
low_df = low_df.dropna(axis=1, how="all")
open_df = open_df.dropna(axis=1, how = "all")
volume_df = volume_df.dropna(axis =1 , how = "all")

valid_cols = (
    close_df.columns.intersection(high_df.columns)
    .intersection(low_df.columns).intersection(open_df.columns).intersection(volume_df.columns)
)
close_df, high_df, low_df, open_df, volume_df = (
    close_df[valid_cols],
    high_df[valid_cols],
    low_df[valid_cols],
    open_df[valid_cols],
    volume_df[valid_cols]
)




[****                   9%                       ]  45 of 500 completed$FDXF: possibly delisted; no price data found  (1d 2015-05-05 -> 2026-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1777953600")
[****************      34%                       ]  169 of 500 completed$VYLR: possibly delisted; no price data found  (1d 2015-05-05 -> 2026-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1777953600")
[**********************47%                       ]  236 of 500 completed$HONA: possibly delisted; no price data found  (1d 2015-05-05 -> 2026-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1777953600")
[*********************100%***********************]  500 of 500 completed

3 Failed downloads:
['FDXF', 'VYLR', 'HONA']: possibly delisted; no price data found  (1d 2015-05-05 -> 2026-05-05) (Yahoo error = "Data doesn't exist for startDate = 1430798400, endDate = 1777953600")


### Bollinger Bands for the S and P 500

In [6]:
macd_dict = {}
signal_dict = {}
hist_dict = {}
rsi_dict = {}
obv_dict = {}
obv_ma_dict = {}
bb_upper_dict = {}
bb_middle_dict = {}
bb_lower_dict = {}

for ticker in close_df.columns:
    prices = close_df[ticker].to_numpy(dtype=np.float64)

    macd, signal, hist = talib.MACD(
        prices,
        fastperiod=12,
        slowperiod=26,
        signalperiod=9
    )

    rsi = talib.RSI(
        prices,
        timeperiod=14
    )

    obv = talib.OBV(close_df[ticker].to_numpy(dtype = np.float64), volume_df[ticker].to_numpy(dtype = np.float64))
    oma = pd.Series(obv, index=close_df.index).rolling(20).mean()

    upper, middle, lower = talib.BBANDS(
        prices,
        timeperiod=20,
        nbdevup=2,
        nbdevdn=2,
        matype=0
    )

    macd_dict[ticker] = macd
    signal_dict[ticker] = signal
    hist_dict[ticker] = hist
    rsi_dict[ticker] = rsi
    obv_dict[ticker] = obv
    obv_ma_dict[ticker] = oma
    bb_upper_dict[ticker] = upper
    bb_middle_dict[ticker] =  middle
    bb_lower_dict[ticker] = lower

macd_df = pd.DataFrame(macd_dict, index=close_df.index)
signal_df = pd.DataFrame(signal_dict, index=close_df.index)
hist_df = pd.DataFrame(hist_dict, index=close_df.index)
rsi_df = pd.DataFrame(rsi_dict, index = close_df.index )
obv_df = pd.DataFrame(obv_dict, index = close_df.index)
oma_df = pd.DataFrame(obv_ma_dict, index = close_df.index)
upper_df = pd.DataFrame(bb_upper_dict, index = close_df.index)
middle_df = pd.DataFrame(bb_middle_dict, index=close_df.index)
lower_df = pd.DataFrame(bb_lower_dict, close_df.index)

entries = (close_df >= middle_df).shift(1).fillna(False).astype(bool)
exits = (close_df < middle_df).shift(1).fillna(False).astype(bool)



backtest1 = vbt.Portfolio.from_signals(
    close=close_df,
    open = open_df,
    high = high_df,
    low = low_df,
    entries=entries,
    exits=exits,
    init_cash=100_000.0,
    size=0.1,                         
    size_type='percent',
    cash_sharing=True,                 
    group_by=True,                     
    freq='1D',
    fees=0.001,
    sl_stop = 0.10,
       
)

print(backtest1.stats())
backtest1.plot().show()

Start                             2015-05-05 00:00:00
End                               2026-05-04 00:00:00
Period                             2766 days 00:00:00
Start Value                                  100000.0
End Value                               203049.978793
Total Return [%]                           103.049979
Benchmark Return [%]                        524.55927
Max Gross Exposure [%]                          100.0
Total Fees Paid                          79893.056146
Max Drawdown [%]                             25.65799
Max Drawdown Duration               573 days 00:00:00
Total Trades                                    78614
Total Closed Trades                             78328
Total Open Trades                                 286
Open Trade PnL                            10958.55823
Win Rate [%]                                39.471198
Best Trade [%]                             451.890664
Worst Trade [%]                            -43.763158
Avg Winning Trade [%]       

### Bullish confirmation using MACD and RSI

In [8]:
entries = ((close_df >= middle_df) & (macd_df >= signal_df) & (rsi_df >= 30)).shift(1).fillna(False).astype(bool)
exits = ((close_df < middle_df) | (macd_df < signal_df) | (rsi_df > 70)).shift(1).fillna(False).astype(bool)



backtest1 = vbt.Portfolio.from_signals(
    close=close_df,
    open = open_df,
    high = high_df,
    low = low_df,
    entries=entries,
    exits=exits,
    init_cash=100_000.0,
    size=0.1,                         
    size_type='percent',
    cash_sharing=True,                 
    group_by=True,                     
    freq='1D',
    fees=0.001,
    sl_stop = 0.10,
       
)

print(backtest1.stats())
backtest1.plot().show()

Start                             2015-05-05 00:00:00
End                               2026-05-04 00:00:00
Period                             2766 days 00:00:00
Start Value                                  100000.0
End Value                               204196.311597
Total Return [%]                           104.196312
Benchmark Return [%]                        524.55927
Max Gross Exposure [%]                          100.0
Total Fees Paid                          85706.214975
Max Drawdown [%]                             29.56714
Max Drawdown Duration               487 days 00:00:00
Total Trades                                    62842
Total Closed Trades                             62621
Total Open Trades                                 221
Open Trade PnL                            9913.828271
Win Rate [%]                                40.786637
Best Trade [%]                             209.965469
Worst Trade [%]                            -43.763158
Avg Winning Trade [%]       

### Using volatility (Bollinger Bands), volume (OBV), trend(MACD) and momentum (RSI) indicators.

In [9]:
entries = ((close_df >= middle_df) & (macd_df >= signal_df) & (rsi_df >= 30) & (obv_df >= oma_df )).shift(1).fillna(False).astype(bool)
exits = ((close_df < middle_df) | (macd_df < signal_df) | (rsi_df > 70) | (obv_df < oma_df)).shift(1).fillna(False).astype(bool)



backtest1 = vbt.Portfolio.from_signals(
    close=close_df,
    open = open_df,
    high = high_df,
    low = low_df,
    entries=entries,
    exits=exits,
    init_cash=100_000.0,
    size=0.1,                         
    size_type='percent',
    cash_sharing=True,                 
    group_by=True,                     
    freq='1D',
    fees=0.001,
    sl_stop = 0.10,
       
)

print(backtest1.stats())
backtest1.plot().show()

Start                            2015-05-05 00:00:00
End                              2026-05-04 00:00:00
Period                            2766 days 00:00:00
Start Value                                 100000.0
End Value                              162777.770901
Total Return [%]                           62.777771
Benchmark Return [%]                       524.55927
Max Gross Exposure [%]                     99.999997
Total Fees Paid                          95174.34384
Max Drawdown [%]                           25.625387
Max Drawdown Duration              521 days 00:00:00
Total Trades                                   70327
Total Closed Trades                            70151
Total Open Trades                                176
Open Trade PnL                           8629.836842
Win Rate [%]                               42.395689
Best Trade [%]                            177.512292
Worst Trade [%]                           -31.391328
Avg Winning Trade [%]                       3.